# Data exploration and cleaning study

This notebook studies the supplied employees, projects and timesheets alongside the modular ETL pipeline. It preserves the source data, profiles quality issues, defines explicit cleaning decisions, and builds **in-memory candidate datasets** with an auditable issue table. It does not overwrite CSVs or load a database.

Run all cells from top to bottom with Python 3 and pandas, with the working directory set to the folder containing this notebook and the three CSVs. If needed, install pandas in your notebook environment (`%pip install pandas`). All results are derived from the files, not from previous notebook state.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd()
COLUMNS = {
    "employees": ["employee_id", "name", "role"],
    "projects": ["project_id", "project_name", "budget"],
    "timesheets": ["employee_id", "project_id", "date", "hours"],
}
raw = {}
for table, columns in COLUMNS.items():
    # Preserve original text, including empty fields; do not infer IDs/numbers/dates.
    frame = pd.read_csv(DATA_DIR / f"{table}.csv", dtype="string", keep_default_na=False)
    assert frame.columns.tolist() == columns, f"Unexpected schema: {table}"
    frame.insert(0, "source_record", range(1, len(frame) + 1))
    raw[table] = frame

# source_record is the 1-based parsed record ordinal, not a physical file line.
display(pd.DataFrame([
    {"table": t, "rows": len(df), "business_columns": len(COLUMNS[t])}
    for t, df in raw.items()
]))
for table, frame in raw.items():
    print(table)
    display(frame.head())

,table,rows,business_columns
0,employees,42,3
1,projects,39,3
2,timesheets,343,4


employees


,source_record,employee_id,name,role
0,1,E001,Sarah Okonkwo,Data Engineer
1,2,E002,James Patel,Data Analyst
2,3,E003,Maria Gonzalez,Backend Engineer
3,4,E004,Tom Whitfield,Product Manager
4,5,E005,Lena Bauer,Data Scientist


projects


,source_record,project_id,project_name,budget
0,1,P001,Alpha Platform Rebuild,120000
1,2,P002,Data Warehouse Migration,85000
2,3,P003,Customer Portal v2,60000
3,4,P004,Internal Analytics Dashboard,45000
4,5,P005,Mobile App Launch,200000


timesheets


,source_record,employee_id,project_id,date,hours
0,1,E001,P001,07/01/2024,8
1,2,E001,P001,08/01/2024,7.5
2,3,E002,P003,07/01/2024,6
3,4,E003,P002,07/01/2024,8
4,5,E004,P004,2024-01-08,4


## 1. Profile completeness, types and duplicates

Read everything as text first: numeric inference could hide bad values and automatic missing-value inference could erase literal strings. Strip surrounding whitespace and represent empty strings as `pd.NA` in a separate working copy. Names keep their accents, spelling and case. IDs are not silently repaired or uppercased.

Duplicate comparisons exclude the provenance column. Separate repeated identical records from **conflicting records sharing an identifier**; keeping the last row is not justified without a version or update timestamp.

In [2]:
prepared = {}
profile = []
for table, source in raw.items():
    frame = source.copy(deep=True)
    for column in COLUMNS[table]:
        trimmed = frame[column].str.strip()
        frame[column] = trimmed.mask(trimmed.eq(""), pd.NA)
        profile.append({
            "table": table, "column": column, "loaded_dtype": str(source[column].dtype),
            "missing_after_trim": int(frame[column].isna().sum()),
            "whitespace_changes": int(source[column].ne(trimmed).sum()),
            "distinct_nonmissing": frame[column].nunique(),
        })
    prepared[table] = frame

display(pd.DataFrame(profile))
for table, frame in prepared.items():
    repeated = frame.duplicated(COLUMNS[table], keep=False)
    print(f"{table}: {frame.duplicated(COLUMNS[table]).sum()} redundant copies")
    display(frame.loc[repeated])

for table, key in [("employees", "employee_id"), ("projects", "project_id")]:
    distinct = prepared[table].drop_duplicates(COLUMNS[table])
    conflicts = distinct[key].notna() & distinct.duplicated(key, keep=False)
    print(f"{table}: conflicting identifiers")
    display(distinct.loc[conflicts])

print("Role vocabulary (no approved role list was supplied)")
display(prepared["employees"]["role"].value_counts(dropna=False).rename("rows").to_frame())

,table,column,loaded_dtype,missing_after_trim,whitespace_changes,distinct_nonmissing
0,employees,employee_id,string,0,0,40
1,employees,name,string,1,0,39
2,employees,role,string,1,0,8
3,projects,project_id,string,1,0,35
4,projects,project_name,string,1,0,35
5,projects,budget,string,1,0,35
6,timesheets,employee_id,string,1,0,42
7,timesheets,project_id,string,0,0,22
8,timesheets,date,string,0,0,106
9,timesheets,hours,string,1,0,18


employees: 2 redundant copies


,source_record,employee_id,name,role
0,1,E001,Sarah Okonkwo,Data Engineer
10,11,E001,Sarah Okonkwo,Data Engineer
14,15,E014,Anouk Vermeer,Data Scientist
31,32,E014,Anouk Vermeer,Data Scientist


projects: 3 redundant copies


,source_record,project_id,project_name,budget
0,1,P001,Alpha Platform Rebuild,120000
6,7,P007,ML Forecasting Pipeline,95000
10,11,P001,Alpha Platform Rebuild,120000
20,21,P007,ML Forecasting Pipeline,95000
28,29,P026,Experimentation Platform,67000
33,34,P026,Experimentation Platform,67000


...

,source_record,employee_id,name,role


projects: conflicting identifiers


,source_record,project_id,project_name,budget


Role vocabulary (no approved role list was supplied)


,rows
role,
Data Engineer,9
Data Analyst,8
Backend Engineer,6
Data Scientist,6
Product Manager,4
Analytics Engineer,4
DevOps Engineer,3
Project Manager,1
<NA>,1


## 2. Study numeric values and date formats

Use numeric conversion only to diagnose failures, never to replace missing or invalid values with zero. The project budget `55000-60000` is a range, not a scalar; choosing its midpoint would invent a business rule. The numeric-only profile below identifies `seven` as text. The final preparation stage explicitly maps the English words `one` through `ten`, case-insensitively, to numbers; other text stays invalid.

Dates contain slash dates, ISO dates and a textual month. **Assumption:** slash dates are day/month/year, supported by dates such as `14/01/2024`; this still needs source-owner confirmation. Match explicit formats rather than asking pandas to guess. The textual-month parser below maps English months explicitly so it does not depend on the machine locale.

In [3]:
def parse_dates(values):
    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    formats = pd.Series("unsupported", index=values.index, dtype="string")
    patterns = [
        (r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y", "day/month/year"),
        (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d", "ISO"),
    ]
    for pattern, fmt, label in patterns:
        mask = values.str.fullmatch(pattern, na=False)
        parsed.loc[mask] = pd.to_datetime(values.loc[mask], format=fmt, errors="coerce")
        formats.loc[mask] = label
    months = {name: f"{n:02}" for n, name in enumerate(
        ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], 1)}
    parts = values.str.extract(r"^(\d{2})-([A-Z][a-z]{2})-(\d{2})$")
    mask = parts[1].isin(months)
    # Explicit study assumption: two-digit years denote 2000–2099.
    iso = "20" + parts.loc[mask, 2] + "-" + parts.loc[mask, 1].map(months) + "-" + parts.loc[mask, 0]
    parsed.loc[mask] = pd.to_datetime(iso, format="%Y-%m-%d", errors="coerce")
    formats.loc[mask] = "English month / two-digit year"
    formats.loc[values.isna()] = "missing"
    return parsed, formats

study = {table: frame.copy(deep=True) for table, frame in prepared.items()}
study["projects"]["budget_numeric"] = pd.to_numeric(study["projects"]["budget"], errors="coerce")
study["timesheets"]["hours_numeric"] = pd.to_numeric(study["timesheets"]["hours"], errors="coerce")
study["timesheets"]["work_date"], date_formats = parse_dates(study["timesheets"]["date"])
display(date_formats.value_counts(dropna=False).rename("rows").to_frame())
for table, column in [("projects", "budget"), ("timesheets", "hours")]:
    frame = study[table]
    numeric = frame[f"{column}_numeric"]
    print(f"{table}: numeric distribution (before validation)")
    display(numeric.describe().to_frame())
    bad = numeric.isna() | numeric.isin([float("inf"), -float("inf")]) | numeric.lt(0)
    if column == "hours":
        bad |= numeric.eq(0) | numeric.gt(24)
    display(frame.loc[bad.fillna(True)])

print("Unparseable dates")
display(study["timesheets"].loc[study["timesheets"]["work_date"].isna()])
print("Observed date range:", study["timesheets"]["work_date"].min(), "to", study["timesheets"]["work_date"].max())

,rows
day/month/year,341
ISO,1
English month / two-digit year,1


projects: numeric distribution (before validation)


,budget_numeric
count,37.0
mean,67270.27027
std,40065.674314
min,-5000.0
25%,40000.0
50%,62000.0
75%,91000.0
max,200000.0


,source_record,project_id,project_name,budget,budget_numeric
7,8,P008,API Gateway Consolidation,55000-60000,<NA>
21,22,P019,Data Quality Framework,-5000,-5000
35,36,P032,Supply Chain Analytics,<NA>,<NA>


timesheets: numeric distribution (before validation)


,hours_numeric
count,340.0
mean,7.232353
std,1.670624
min,-3.0
25%,6.5
50%,7.5
75%,8.0
max,25.0


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
61,62,E014,P007,29/01/2024,-3,-3.0,2024-01-29
67,68,E020,P003,30/01/2024,25,25.0,2024-01-30
202,203,E005,P007,31/03/2024,<NA>,<NA>,2024-03-31
272,273,E020,P003,01/05/2024,not_a_number,<NA>,2024-05-01
330,331,E030,P009,27/05/2024,seven,<NA>,2024-05-27


Unparseable dates


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date


Observed date range: 2024-01-07 00:00:00 to 2024-05-30 00:00:00


## 3. Study references and the timesheet grain

Expected entities are employees and projects, connected by timesheets. Missing references must be checked against the original known identifiers **and** the eventual accepted dimensions: a known project can still be held for review.

There is no timesheet entry ID. For this study, assume one record per employee, project and day. Identical repeated entries are treated as duplicate ingestion; conflicting hours at the same grain require review. Confirm this assumption before a production uniqueness constraint: multiple legitimate entries per day would require a source entry ID instead.

In [4]:
times = study["timesheets"]
for column, parent in [("employee_id", "employees"), ("project_id", "projects")]:
    known = prepared[parent][column].dropna()
    missing_reference = times[column].notna() & ~times[column].isin(known)
    print(f"Unknown {column} values")
    display(times.loc[missing_reference])

GRAIN = ["employee_id", "project_id", "work_date"]
semantic_columns = GRAIN + ["hours_numeric"]
parseable = times.loc[times[semantic_columns].notna().all(axis=1)]
print("Repeated employee/project/day groups (includes identical copies)")
display(parseable.loc[parseable.duplicated(GRAIN, keep=False)].sort_values(GRAIN))
print("Daily hours before final validation, excluding identical parsed copies")
# Descriptive exploration only: showing days above 12 does not flag or reject them.
daily_study = (parseable.drop_duplicates(semantic_columns)
               .groupby(["employee_id", "work_date"], as_index=False)["hours_numeric"].sum())
display(daily_study.loc[daily_study["hours_numeric"].gt(12)])

Unknown employee_id values


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
56,57,E099,P001,25/01/2024,8,8.0,2024-01-25
200,201,E050,P001,31/03/2024,8,8.0,2024-03-31


Unknown project_id values


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
57,58,E010,P999,28/01/2024,7.5,7.5,2024-01-28
201,202,E001,P999,31/03/2024,6,6.0,2024-03-31


Repeated employee/project/day groups (includes identical copies)


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date
156,157,E001,P001,11/03/2024,7.5,7.5,2024-03-11
159,160,E001,P001,11/03/2024,7.5,7.5,2024-03-11
5,6,E005,P007,08/01/2024,9,9.0,2024-01-08
6,7,E005,P007,08/01/2024,9,9.0,2024-01-08


Daily hours before final validation, excluding identical parsed copies


,employee_id,work_date,hours_numeric
183,E020,2024-01-30,25.0


## 4. Final cleaning policy

The exploration above profiles raw values. The implementation below uses the agreed policy in `pipeline/`, shared with the command-line runner so this notebook does not maintain a second set of cleaning rules.

| Check | Handling |
|---|---|
| Whitespace and empty strings | Trim and normalize nulls on copies; preserve source values. |
| Identifiers | Require E/P followed by three ASCII digits; reject missing or malformed IDs. |
| Identical dimension records | Keep first; reject redundant copies. |
| Conflicting dimension IDs | Hold distinct versions for review; exclude unresolved IDs. |
| Missing name, role or project name | Review but retain unique valid ID with a null attribute. |
| Invalid, missing or range-valued budget | Review and use null; do not invent an amount. |
| Hour words | Map exact lowercase matches `one` through `ten` to 1–10 after trimming. |
| Other invalid hours | Reject missing, nonnumeric, nonfinite or out-of-range hours; require 0 < hours <= 24. |
| Dates | Parse the three observed formats explicitly, using day-first slash dates and years 2000–2099 for two-digit years. |
| Unknown references | Reject. Hold references to unresolved conflicting dimension IDs for review. |
| Duplicate parsed timesheets | Retain first identical employee/project/day/hours entry. |
| Conflicting timesheet grain | Hold the affected employee/day for review, deferring its daily total. |
| Daily hours | After excluding hard failures and copies, reject all contributing entries if the unambiguous total exceeds 24. |

The assumed grain is one entry per employee/project/day. Missing descriptions and budgets do not block valid work. There is no 12-hour rejection/review rule.

Each record gets one final status, with rejection taking precedence over review. Reviewed dimensions can enter clean output only for nullable attribute issues, identified by `include_in_clean`. Reviewed timesheets never enter summaries.


In [5]:
from pipeline.prepare import prepare
from pipeline.validate import validate, verify_outputs
from pipeline.transform import transform

pipeline_raw, pipeline_prepared = prepare(DATA_DIR)
classified, validation_issues = validate(pipeline_raw, pipeline_prepared)
clean_outputs = transform(classified)
verify_outputs(pipeline_raw, classified, clean_outputs)

print("Issue counts (a source record can have multiple issues)")
display(validation_issues.groupby(["source_table", "rule", "status"]).size().rename("issues").reset_index())
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(validation_issues)


Issue counts (a source record can have multiple issues)


,source_table,rule,status,issues
0,employees,duplicate_record,rejected,2
1,employees,missing_description,review,2
2,projects,duplicate_record,rejected,3
3,projects,invalid_budget,review,3
4,projects,invalid_id,rejected,1
5,projects,missing_description,review,1
6,timesheets,duplicate_record,rejected,2
7,timesheets,invalid_hours,rejected,4
8,timesheets,invalid_id,rejected,1
9,timesheets,unknown_reference,rejected,4


,source_table,source_record,column,raw_value,rule,status,reason
0,employees,11,employee_id,E001,duplicate_record,rejected,Identical normalized record; retain first source record.
1,employees,14,name,,missing_description,review,Keep identifiable record with a null attribute; request correction.
2,employees,22,role,,missing_description,review,Keep identifiable record with a null attribute; request correction.
3,employees,32,employee_id,E014,duplicate_record,rejected,Identical normalized record; retain first source record.
4,projects,8,budget,55000-60000,invalid_budget,review,Budget must be a finite nonnegative scalar; use null until corrected.
5,projects,11,project_id,P001,duplicate_record,rejected,Identical normalized record; retain first source record.
6,projects,14,project_name,,missing_description,review,Keep identifiable record with a null attribute; request correction.
7,projects,15,project_id,,invalid_id,rejected,Missing identifier or invalid format; expected prefix plus three digits.
8,projects,21,project_id,P007,duplicate_record,rejected,Identical normalized record; retain first source record.
9,projects,22,budget,-5000,invalid_budget,review,Budget must be a finite nonnegative scalar; use null until corrected.


## 5. Clean outputs, review records and reconciliation

This section executes the same functions as the runner without exporting files. Every source record must be accounted for. Clean dimension counts include usable reviewed records, so clean counts can differ from accepted-status counts.


In [6]:
reconciliation = pd.DataFrame([
    {"table": table, "raw": len(pipeline_raw[table]),
     **{status: int(frame["status"].eq(status).sum()) for status in ["accepted", "review", "rejected"]},
     "clean": int(frame["include_in_clean"].sum())}
    for table, frame in classified.items()
]).set_index("table")
assert reconciliation[["accepted", "review", "rejected"]].sum(axis=1).eq(reconciliation["raw"]).all()
display(reconciliation)
clean_timesheets = clean_outputs["timesheets_clean"]
print("Clean timesheets:", len(clean_timesheets), "rows;", clean_timesheets["hours"].sum(), "hours")
display(clean_timesheets.head(10))
display(clean_outputs["hours_by_project"])
display(clean_outputs["hours_by_employee"])
for table, frame in classified.items():
    print(f"{table}: review records")
    display(frame.loc[frame["status"].eq("review")])
print("All output integrity and reconciliation checks passed.")


,raw,accepted,review,rejected,clean
table,,,,,
employees,42,38,2,2,40
projects,39,31,4,4,35
timesheets,343,332,0,11,332


Clean timesheets: 332 rows; 2390.0 hours


,employee_id,project_id,date,hours
0,E001,P001,2024-01-07,8.0
1,E001,P001,2024-01-08,7.5
2,E001,P001,2024-02-01,8.0
3,E001,P001,2024-03-11,7.5
4,E001,P001,2024-04-28,8.0
5,E001,P002,2024-01-15,8.0
6,E001,P005,2024-01-23,8.0
7,E001,P005,2024-03-26,8.0
8,E001,P005,2024-05-09,8.0
9,E001,P007,2024-03-05,8.0


,project_id,project_name,total_hours,timesheet_count
0,P001,Alpha Platform Rebuild,191.0,25
1,P002,Data Warehouse Migration,107.0,14
2,P003,Customer Portal v2,141.0,20
3,P004,Internal Analytics Dashboard,80.5,19
4,P005,Mobile App Launch,132.5,18
5,P006,Legacy System Decommission,98.5,13
6,P007,ML Forecasting Pipeline,310.0,38
7,P008,API Gateway Consolidation,113.5,15
8,P009,Compliance Reporting Tool,100.5,14
9,P010,Data Catalogue Implementation,129.0,18


,employee_id,name,total_hours,timesheet_count
0,E001,Sarah Okonkwo,125.0,16
1,E002,James Patel,82.0,12
2,E003,Maria Gonzalez,99.0,13
3,E004,Tom Whitfield,26.5,6
4,E005,Lena Bauer,95.0,11
5,E006,Chris Ndukwe,78.0,10
6,E007,Priya Sharma,75.0,10
7,E008,Oliver Marsh,67.5,10
8,E009,Fatima Al-Hassan,46.0,6
9,E010,Daniel Kovač,60.0,8


...

projects: review records


,source_record,project_id,project_name,budget,budget_numeric,status,include_in_clean
7,8,P008,API Gateway Consolidation,55000-60000,<NA>,review,True
13,14,P013,<NA>,15000,15000,review,True
21,22,P019,Data Quality Framework,-5000,<NA>,review,True
35,36,P032,Supply Chain Analytics,<NA>,<NA>,review,True


timesheets: review records


,source_record,employee_id,project_id,date,hours,hours_numeric,work_date,status,include_in_clean


All output integrity and reconciliation checks passed.


## 6. Findings and implementation handoff

The supplied files contain redundant duplicates, missing dimension descriptions, missing/negative/range-valued budgets, unknown timesheet references, missing identifiers, mixed date formats and invalid hours. Explicit word mapping recovers the `seven` entry without accepting arbitrary text.

The current policy produces **40 clean employees, 35 clean projects and 332 clean timesheets totaling 2,390 hours**. Eleven timesheets are rejected. Two employees and four projects need attribute review but remain usable as dimensions. The executed tables above provide the per-record reasons and accounting.

Confirm the date interpretation, positive-hours assumption, timesheet grain and nullable dimension policy with a source owner. No labor cost or budget utilization is inferred without rates and currency information.

The modular implementation separates preparation, validation, transformation and output orchestration. Run `python -m pipeline.run` to generate the deliverables. See `README.md` for execution, `architecture.md` for design decisions, and `diagram.md` / `schema.sql` for the relational model.

This notebook only builds in-memory results. Restart and run all cells from the repository root for repeatability. The runner overwrites fixed CSV outputs instead of appending; interrupted export requires a rerun. Atomic publication, incremental loads and a review application are future improvements outside this assessment.
